# Normalizacao do Relatorio de Despesas do Sistema ATUA

Este notebook le o arquivo `Relatorio_Despesas_Sistema-ATUA_{MM}.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_{AAAA}_{MM}.xlsx`).

Diferente do ATUA, o SAGI classifica a GSL na divisao **TRANSMOVE GSL (1.4)** com quatro filiais e dois departamentos analiticos (TRANSPORTE / ADMINISTRATIVO). Este notebook:

1. Ajuste o parametro `MES_REFERENCIA` na primeira celula de codigo (formato `MM/AAAA`). Para relatorios consolidados de varios meses (ex.: `Relatorio_Despesas_Sistema-ATUA_Jan-Fev.xls`), preencha tambem `PERIODO_ARQUIVO = "Jan-Fev"` na mesma celula
2. Le os dados da aba `base`
3. Aplica a **dinamica de tratamento de despesas**:
   - Exclui historicos `{225, 230, 231, 237, 238}` (Remessa, Devolucao de peca, Pamcard, Center Pecas & afins, Transferencias entre filiais)
   - Exclui linhas com `nm_unidade_centro_custo` contendo `CUSTO JA ALOCADO` (evita duplicidade com nota-mae)
4. Converte o Centro de Custo do ATUA (`cd_unidade` + `cd_centro_custo`) para o padrao SAGI
5. Converte o Plano de Contas do ATUA (`cd_historico` / `nm_historico`) para o padrao SAGI
6. Gera um Excel no layout FECHAMENTO_ODBC

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = "02/2026"   # formato MM/AAAA — base do modelo de fechamento

# Opcional: sufixo textual do arquivo de entrada/saida quando o relatorio
# NAO for de um unico mes numerico. Se None, usa o mes numerico (ex: "03").
# Exemplo: PERIODO_ARQUIVO = "Jan-Fev" para o arquivo
#   Relatorio_Despesas_Sistema-ATUA_Jan-Fev.xls
# gerara como saida: ATUA_despesas_fechamento_Jan-Fev.xlsx
PERIODO_ARQUIVO = None

# Opcional: mes/ano do arquivo-modelo FECHAMENTO_ODBC a usar como referencia
# de colunas. Se None, usa o mesmo mes de MES_REFERENCIA. Defina aqui um mes
# que ja exista em disco quando o modelo do mes atual ainda nao foi gerado.
# Exemplo: MODELO_MES = "03/2026"  ->  usa FECHAMENTO_ODBC_2026_03.xlsx
MODELO_MES = None
# ────────────────────────────────────────────────────────────────────────────

mes_num, ano = MES_REFERENCIA.split("/")
sufixo_entrada = PERIODO_ARQUIVO if PERIODO_ARQUIVO else mes_num
sufixo_saida   = PERIODO_ARQUIVO if PERIODO_ARQUIVO else f"{mes_num}-{ano}"

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ARQUIVO_ENTRADA = ATUA_DIR / f"Relatorio_Despesas_Sistema-ATUA_{sufixo_entrada}.xls"
ARQUIVO_SAIDA   = ATUA_DIR / f"ATUA_despesas_fechamento_{sufixo_saida}.xlsx"

if MODELO_MES:
    _mod_mes, _mod_ano = MODELO_MES.split("/")
    ARQUIVO_MODELO_FECHAMENTO = REFS_DIR / f"FECHAMENTO_ODBC_{_mod_ano}_{_mod_mes}.xlsx"
else:
    ARQUIVO_MODELO_FECHAMENTO = REFS_DIR / f"FECHAMENTO_ODBC_{ano}_{mes_num}.xlsx"

if not ARQUIVO_MODELO_FECHAMENTO.exists():
    _candidatos = sorted(REFS_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
    if _candidatos:
        ARQUIVO_MODELO_FECHAMENTO = _candidatos[-1]
        print(f"[AVISO] Modelo para {ano}/{mes_num} nao encontrado. Usando como referencia: {ARQUIVO_MODELO_FECHAMENTO.name}")
    else:
        raise FileNotFoundError(f"Nenhum arquivo FECHAMENTO_ODBC_*.xlsx encontrado em {REFS_DIR.resolve()}")

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

COLUNAS_INTERESSE = [
    "dt_lancamento",
    "nm_pessoa_favorecido",
    "cd_historico",
    "nm_historico",
    "nm_pessoa_filial",
    "dt_lancamento_",
    "vl_lancamento",
    "vl_lancamento_liquido",
    "ds_complemento",
    "cd_centro_custo",
    "nm_centro_custo",
    "cd_unidade",
    "nm_unidade",
    "nm_unidade_centro_custo",
    "nr_documento",
]

import xlrd as _xlrd
_wb = _xlrd.open_workbook(ARQUIVO_ENTRADA)
_abas = _wb.sheet_names()
if "base" in _abas:
    _aba_leitura = "base"
else:
    _aba_leitura = _abas[0]
    print(f"[AVISO] Aba 'base' nao encontrada. Usando a primeira aba disponivel: '{_aba_leitura}'")
    print(f"        Abas encontradas: {_abas}")

df_atua = pd.read_excel(ARQUIVO_ENTRADA, sheet_name=_aba_leitura, header=0, engine="xlrd", dtype=object)
df_atua = df_atua[COLUNAS_INTERESSE].copy()

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas lidas: {len(df_atua)}")
print(f"Colunas usadas: {df_atua.columns.tolist()}")
df_atua.head(5)

[AVISO] Modelo para 2026/02 nao encontrado. Usando como referencia: FECHAMENTO_ODBC_2026_03.xlsx
[AVISO] Aba 'base' nao encontrada. Usando a primeira aba disponivel: 'Planilha 1'
        Abas encontradas: ['Planilha 1']
Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Relatorio_Despesas_Sistema-ATUA_Jan-Fev.xls
Linhas lidas: 331
Colunas usadas: ['dt_lancamento', 'nm_pessoa_favorecido', 'cd_historico', 'nm_historico', 'nm_pessoa_filial', 'dt_lancamento_', 'vl_lancamento', 'vl_lancamento_liquido', 'ds_complemento', 'cd_centro_custo', 'nm_centro_custo', 'cd_unidade', 'nm_unidade', 'nm_unidade_centro_custo', 'nr_documento']


,dt_lancamento,nm_pessoa_favorecido,cd_historico,nm_historico,nm_pessoa_filial,dt_lancamento_,vl_lancamento,vl_lancamento_liquido,ds_complemento,cd_centro_custo,nm_centro_custo,cd_unidade,nm_unidade,nm_unidade_centro_custo,nr_documento
0,2026-01-02 11:07:42.76648,AUTO POSTO SANTA TEREZINHA DE AVARE LTDA,231,DIESEL - PAMCARD,GSL PRUDENTE,2026-01-02 10:56:44,826.54,826.54,"TRIB APROX R$: 71,08 FEDERAL, 151,26 ESTADUAL;...",95,Sucata,9,PRUDENTE ADMINISTRATIVO/COMERCIAL,9 / PRUDEN - 95 / Sucata,707953
1,2026-01-02 11:07:43.196317,IBIRAREMA COMERCIO DE COMBUSTIVEIS LTDA,231,DIESEL - PAMCARD,GSL PRUDENTE,2026-01-02 10:56:44,1113.56,1113.56,"REFERENTE AOS DOCUMENTOS: NFC-E SERIE 1, NUM. ...",95,Sucata,9,PRUDENTE ADMINISTRATIVO/COMERCIAL,9 / PRUDEN - 95 / Sucata,28760
2,2026-01-02 11:07:43.312911,AUTO POSTO PHOENIX PINHAL DE ITAPETININGA LTDA,231,DIESEL - PAMCARD,GSL PRUDENTE,2026-01-02 10:56:44,2550.03,2550.03,"TRIB APROX R$: 219,30 FEDERAL, 466,66 ESTADUAL...",95,Sucata,9,PRUDENTE ADMINISTRATIVO/COMERCIAL,9 / PRUDEN - 95 / Sucata,43165
3,2026-01-02 11:07:43.41559,ROZINELI & ROZINELI COMERCIO DE COMBUSTIVEL LTDA,231,DIESEL - PAMCARD,GSL PRUDENTE,2026-01-02 10:56:44,879.97,879.97,ICMS MONOFASICO SOBRE COMBUSTIVEIS COBRADO ANT...,95,Sucata,9,PRUDENTE ADMINISTRATIVO/COMERCIAL,9 / PRUDEN - 95 / Sucata,62429
4,2026-01-02 11:07:43.578948,POSTO TREVAO DE PIRAJU LTDA,231,DIESEL - PAMCARD,GSL PRUDENTE,2026-01-02 10:56:44,945.38,945.38,"REFERENTE AOS DOCUMENTOS: NFC-E SERIE 2, NUM. ...",95,Sucata,9,PRUDENTE ADMINISTRATIVO/COMERCIAL,9 / PRUDEN - 95 / Sucata,26653


In [29]:
## Filtro de historicos a desconsiderar (Dinamica)
#
# Codigos excluidos conforme dinamica de tratamento do relatorio de despesas:
#   225 - Remessa
#   230 - Devolucao de peca (analisar todas)
#   231 - Pamcard (combustivel ja lancado via SAGI)
#   237 - Center Pecas / Distribuidora Automotiva / Odapel / Pellegrino
#   238 - Transferencias entre filiais
APLICAR_FILTRO_HISTORICO = True
CD_HISTORICO_EXCLUIR = {225, 230, 231, 237, 238}

def _cd_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

if APLICAR_FILTRO_HISTORICO:
    mask_excluir = df_atua["cd_historico"].apply(lambda v: _cd_int(v) in CD_HISTORICO_EXCLUIR)
    n_excluidas = int(mask_excluir.sum())

    if n_excluidas:
        print(f"Historicos desconsiderados: {sorted(CD_HISTORICO_EXCLUIR)}")
        print("Linhas excluidas:")
        print(df_atua.loc[mask_excluir, ["cd_historico", "nm_historico", "nm_pessoa_filial", "vl_lancamento"]].to_string(index=True))
        df_atua = df_atua[~mask_excluir].reset_index(drop=True)
        print(f"\nLinhas restantes para processamento: {len(df_atua)}")
    else:
        print(f"Nenhuma linha com cd_historico em {sorted(CD_HISTORICO_EXCLUIR)} encontrada.")
        print(f"Linhas para processamento: {len(df_atua)}")
else:
    print("Filtro de historico desativado (APLICAR_FILTRO_HISTORICO=False).")
    print(f"Linhas para processamento: {len(df_atua)}")

Historicos desconsiderados: [225, 230, 231, 237, 238]
Linhas excluidas:
    cd_historico      nm_historico nm_pessoa_filial vl_lancamento
0            231  DIESEL - PAMCARD     GSL PRUDENTE        826.54
1            231  DIESEL - PAMCARD     GSL PRUDENTE       1113.56
2            231  DIESEL - PAMCARD     GSL PRUDENTE       2550.03
3            231  DIESEL - PAMCARD     GSL PRUDENTE        879.97
4            231  DIESEL - PAMCARD     GSL PRUDENTE        945.38
5            231  DIESEL - PAMCARD     GSL PRUDENTE        967.07
6            231  DIESEL - PAMCARD     GSL PRUDENTE       1002.66
7            231  DIESEL - PAMCARD     GSL PRUDENTE       1139.24
8            231  DIESEL - PAMCARD     GSL PRUDENTE         975.8
9            231  DIESEL - PAMCARD     GSL PRUDENTE        870.15
10           231  DIESEL - PAMCARD     GSL PRUDENTE       1345.14
11           231  DIESEL - PAMCARD     GSL PRUDENTE        866.84
12           231  DIESEL - PAMCARD     GSL PRUDENTE        987.68
13  

## Filtro de linhas com "CUSTO JA ALOCADO"

Linhas cuja `nm_unidade_centro_custo` contem `CUSTO JA ALOCADO` representam custos
que ja foram alocados a outra filial/centro e nao devem ser contabilizados novamente
no fechamento SAGI (evita duplicidade). Exemplos comuns: `2 / CUSTO JA ALOCADO - NOTA MAE`,
`3 / FROTA - 2 / CUSTO JA ALOCADO - NOTA MAE`.

In [30]:
FILTRAR_CUSTO_JA_ALOCADO = True

if FILTRAR_CUSTO_JA_ALOCADO:
    mask_alocado = df_atua["nm_unidade_centro_custo"].astype(str).str.contains(
        "CUSTO JA ALOCADO", case=False, na=False
    )
    n_alocado = int(mask_alocado.sum())

    if n_alocado:
        print(f"[EXCLUIDO] {n_alocado} linha(s) com 'CUSTO JA ALOCADO' em nm_unidade_centro_custo:")
        print(
            df_atua.loc[
                mask_alocado,
                ["cd_unidade", "nm_unidade_centro_custo", "vl_lancamento_liquido"],
            ].to_string(index=True)
        )
        df_atua = df_atua[~mask_alocado].reset_index(drop=True)
        print(f"\nLinhas restantes para processamento: {len(df_atua)}")
    else:
        print("Nenhuma linha com 'CUSTO JA ALOCADO' encontrada.")
        print(f"Linhas para processamento: {len(df_atua)}")
else:
    print("Filtro 'CUSTO JA ALOCADO' desativado (FILTRAR_CUSTO_JA_ALOCADO=False).")
    print(f"Linhas para processamento: {len(df_atua)}")

Nenhuma linha com 'CUSTO JA ALOCADO' encontrada.
Linhas para processamento: 140


## Mapa de Centros de Custo (ATUA -> SAGI)

O ATUA classifica o CC com dois codigos: `cd_unidade` (filial) + `cd_centro_custo` (departamento).
No SAGI, a GSL e a divisao **1.4 TRANSMOVE GSL**, com 4 filiais:

| cd_unidade ATUA | Filial ATUA | Filial SAGI | Codigo SAGI nivel 3 |
|---|---|---|---|
| 8 | FROTA TERCEIRO PRUDENTE | PRESIDENTE PRUDENTE | 1.4.1 |
| 9 | PRUDENTE ADMINISTRATIVO/COMERCIAL | PRESIDENTE PRUDENTE | 1.4.1 |
| 12 | MARINGA ADMINISTRATIVO/COMERCIAL | MARINGA | 1.4.3 |
| 16 | DOURADOS ADMINISTRATIVO/COMERCIAL | DOURADOS | 1.4.2 |
| 26 | BARUERI ADMINISTRATIVO/COMERCIAL | BARUERI | 1.4.4 |

E dois departamentos analiticos:

| cd_centro_custo ATUA | Nome ATUA | Departamento SAGI | Sufixo codigo |
|---|---|---|---|
| 81 | Transporte | TRANSPORTE | .1 |
| 95 | Sucata | TRANSPORTE | .1 |
| 100 | ADMINISTRATIVO/COMERCIAL | ADMINISTRATIVO | .2 |
| 102 | FROTA TERCEIRO PRUDENTE | TRANSPORTE | .1 |

In [31]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

def _int_str(v) -> str:
    """Converte floats como 9.0 para '9' (o pandas as vezes le inteiros como float)."""
    if pd.isna(v):
        return ""
    try:
        f = float(v)
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return _str(v)

print("=== Valores unicos de filial/unidade no ATUA ===")
print(df_atua[["cd_unidade", "nm_unidade"]].drop_duplicates().sort_values("cd_unidade").to_string(index=False))
print()
print("=== Valores unicos de departamento no ATUA ===")
print(df_atua[["cd_centro_custo", "nm_centro_custo"]].drop_duplicates().sort_values("cd_centro_custo").to_string(index=False))

MAPA_FILIAL = {
    "8":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "9":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "12": {"n3_cod": "1.4.3", "n3_desc": "MARINGA",             "filial_saida": "GSL MARINGA"},
    "16": {"n3_cod": "1.4.2", "n3_desc": "DOURADOS",            "filial_saida": "GSL DOURADOS"},
    "26": {"n3_cod": "1.4.4", "n3_desc": "BARUERI",             "filial_saida": "GSL BARUERI"},
}

MAPA_DEPARTAMENTO = {
    "81":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "95":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "100": {"sufixo": "2", "desc": "ADMINISTRATIVO"},
    "102": {"sufixo": "1", "desc": "TRANSPORTE"},
}

def mapear_cc(cd_unidade, cd_centro_custo) -> dict | None:
    """Retorna o codigo SAGI completo + toda a hierarquia ou None se nao for mapeavel."""
    uni = _int_str(cd_unidade)
    dep = _int_str(cd_centro_custo)
    filial = MAPA_FILIAL.get(uni)
    depto = MAPA_DEPARTAMENTO.get(dep)
    if not filial or not depto:
        return None
    cod_n4 = f"{filial['n3_cod']}.{depto['sufixo']}"
    return {
        "n1_cod": "1.4",
        "n1_desc": "DESPESA",
        "n2_cod": "1.4",
        "n2_desc": "TRANSMOVE GSL",
        "n3_cod": filial["n3_cod"],
        "n3_desc": filial["n3_desc"],
        "n4_cod": cod_n4,
        "n4_desc": depto["desc"],
        "filial_saida": filial["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

ccs_nao_mapeados_cc = []
for _, r in df_atua[["cd_unidade", "cd_centro_custo", "nm_unidade", "nm_centro_custo"]].drop_duplicates().iterrows():
    if mapear_cc(r["cd_unidade"], r["cd_centro_custo"]) is None:
        ccs_nao_mapeados_cc.append(
            (_int_str(r["cd_unidade"]), _str(r["nm_unidade"]), _int_str(r["cd_centro_custo"]), _str(r["nm_centro_custo"]))
        )

if ccs_nao_mapeados_cc:
    print("\n[ALERTA] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")
else:
    print("\nTodos os centros de custo foram mapeados com sucesso.")

=== Valores unicos de filial/unidade no ATUA ===
cd_unidade                        nm_unidade
         8           FROTA TERCEIRO PRUDENTE
         9 PRUDENTE ADMINISTRATIVO/COMERCIAL
        12  MARINGA ADMINISTRATIVO/COMERCIAL
        16 DOURADOS ADMINISTRATIVO/COMERCIAL
        26  BARUERI ADMINISTRATIVO/COMERCIAL

=== Valores unicos de departamento no ATUA ===
cd_centro_custo          nm_centro_custo
            100 ADMINISTRATIVO/COMERCIAL
            102  FROTA TERCEIRO PRUDENTE

Todos os centros de custo foram mapeados com sucesso.


## Mapa de Plano de Contas (ATUA -> SAGI)

O ATUA usa `cd_historico` (codigo) e `nm_historico` (descricao) como plano de contas. Esta celula mapeia cada historico ATUA para o codigo oficial do SAGI (`02-Referencias/Plano de Contas.pdf`):

| cd_historico ATUA | nm_historico ATUA | Codigo SAGI | Descricao SAGI |
|---|---|---|---|
| 15 | ENERGIA ELETRICA | 7.5.2 | ENERGIA ELETRICA |
| 16 | ALUGUEL E CONDOMINIOS | 7.5.31 | ALUGUEL ADMINISTRATIVO |
| 17 | SEGURO DE CARGAS | 6.6.4 | SEGURO DE CARGAS |
| 23 | TARIFAS BANCARIAS | 7.5.22 | DESPESAS BANCARIAS |
| 26 | ASSESSORIAS E TELECONSULTAS | 7.5.9 | CONSULTORIA |
| 46 | IMPOSTOS E TAXAS DIVERSAS | 7.5.17 | TAXAS |
| 62 | FRETES PAGOS | 6.6.1 | FRETE DE TERCEIROS |
| 69 | HONORARIOS CONTABEIS | 7.5.7 | HONORARIOS CONTABEIS |
| 95 | ICMS | 7.4.12 | ICMS |
| 209 | PESSOAL - INSS PATRONAL | 7.3.3 | INSS |
| 229 | PESSOAL - PRO LABORE | 7.3.12 | PRO LABORE |
| 231 | DIESEL - PAMCARD | 7.1.4 | COMBUSTIVEL - DIESEL (POSTO) |

In [32]:
print("=== Valores unicos de Plano de Contas no ATUA ===")
print(df_atua[["cd_historico", "nm_historico"]].drop_duplicates().sort_values("cd_historico").to_string(index=False))

MAPA_PLANO_CONTAS = {
    "15":  {"cod": "7.5.2",  "desc": "ENERGIA ELETRICA"},
    "16":  {"cod": "7.5.31", "desc": "ALUGUEL ADMINISTRATIVO"},
    "17":  {"cod": "6.6.4",  "desc": "SEGURO DE CARGAS"},
    "23":  {"cod": "7.5.22", "desc": "DESPESAS BANCARIAS"},
    "26":  {"cod": "7.5.9",  "desc": "CONSULTORIA"},
    "46":  {"cod": "7.5.17", "desc": "TAXAS"},
    "62":  {"cod": "6.6.1",  "desc": "FRETE DE TERCEIROS"},
    "69":  {"cod": "7.5.7",  "desc": "HONORARIOS CONTABEIS"},
    "95":  {"cod": "7.4.12", "desc": "ICMS"},
    "209": {"cod": "7.3.3",  "desc": "INSS"},
    "229": {"cod": "7.3.12", "desc": "PRO LABORE"},
    "231": {"cod": "7.1.4",  "desc": "COMBUSTIVEL - DIESEL (POSTO)"},
}

def mapear_plano_contas(cd_historico) -> dict | None:
    key = _int_str(cd_historico)
    return MAPA_PLANO_CONTAS.get(key)

historicos_nao_mapeados = []
for _, r in df_atua[["cd_historico", "nm_historico"]].drop_duplicates().iterrows():
    if mapear_plano_contas(r["cd_historico"]) is None:
        historicos_nao_mapeados.append((_int_str(r["cd_historico"]), _str(r["nm_historico"])))

if historicos_nao_mapeados:
    print("\n[ALERTA] Planos de Contas nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")
else:
    print("\nTodos os planos de contas foram mapeados com sucesso.")

=== Valores unicos de Plano de Contas no ATUA ===
cd_historico                nm_historico
          15            ENERGIA ELETRICA
          16       ALUGUEL E CONDOMINIOS
          17            SEGURO DE CARGAS
          23           TARIFAS BANCARIAS
          26 ASSESSORIAS E TELECONSULTAS
          31              MULTA - OUTRAS
          46   IMPOSTOS E TAXAS DIVERSAS
          62                FRETES PAGOS
          69        HONORARIOS CONTABEIS
          95                        ICMS
         209     PESSOAL - INSS PATRONAL
         229        PESSOAL - PRO LABORE

[ALERTA] Planos de Contas nao mapeados:
  cd_historico=31 (MULTA - OUTRAS)


## Conversao para layout FECHAMENTO_ODBC

Regras de mapeamento aplicadas (conforme dinamica de despesas):

- `nm_pessoa_filial` -> `filial`
- `dt_lancamento_` -> `data_nf` (data do lancamento financeiro)
- `data_pagamento` -> vazio (nao mapeado pela dinamica)
- `vl_lancamento_liquido` -> `valor_nf`, `valor_pago`, `valor_conta`
- `nm_pessoa_favorecido` -> `credor_forn_cli_func`
- `nm_centro_custo` -> `observacao`
- `nm_historico` -> `Dados auxiliares`
- `cd_unidade` + `cd_centro_custo` -> `n1_cod_centro_custo` ate `n4_cod_centro_custo` + descricoes hierarquicas
- `cd_historico` -> `cod_conta` + `conta` + `cod_conta-descr`
- `nr_documento` -> `titulo`
- `Origem` = "Saidas (Aplicacoes)" (fixo — todo o relatorio ATUA de despesas e saida)
- `Sistema` = "ATUA"

In [33]:
def _to_float(v):
    if pd.isna(v):
        return pd.NA
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace("R$", "").replace(" ", "")
    if not s:
        return pd.NA
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return pd.NA

def _fmt_brl(v):
    if pd.isna(v):
        return ""
    s = f"{float(v):,.2f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def _fmt_data(v) -> str:
    """Formata datetimes do pandas como dd/mm/aaaa. Se ja for string, mantem."""
    if pd.isna(v) or v == "":
        return ""
    try:
        ts = pd.to_datetime(v, errors="coerce")
        if pd.isna(ts):
            return str(v).strip()
        return ts.strftime("%d/%m/%Y")
    except Exception:
        return str(v).strip()

modelo_cols = pd.read_excel(ARQUIVO_MODELO_FECHAMENTO, nrows=0).columns.tolist()
print("Colunas do modelo FECHAMENTO:", modelo_cols)

linhas_saida = []
alertas_linhas_cc = []
alertas_linhas_pc = []

for i, row in df_atua.iterrows():
    cc_map = mapear_cc(row["cd_unidade"], row["cd_centro_custo"])
    pc_map = mapear_plano_contas(row["cd_historico"])

    if cc_map is None:
        alertas_linhas_cc.append({
            "idx": i,
            "cd_unidade": _int_str(row["cd_unidade"]),
            "nm_unidade": _str(row["nm_unidade"]),
            "cd_centro_custo": _int_str(row["cd_centro_custo"]),
            "nm_centro_custo": _str(row["nm_centro_custo"]),
        })
    if pc_map is None:
        alertas_linhas_pc.append({
            "idx": i,
            "cd_historico": _int_str(row["cd_historico"]),
            "nm_historico": _str(row["nm_historico"]),
        })

    valor = _to_float(row["vl_lancamento_liquido"])

    nova = {c: pd.NA for c in modelo_cols}
    nova["id"] = i + 1

    if cc_map is not None:
        nova["Segmento"] = cc_map["segmento"]
        nova["n1_cod_centro_custo"] = cc_map["n1_cod"]
        nova["n1_centro_custo"] = cc_map["n1_desc"]
        nova["n1_CC"] = f"{cc_map['n1_cod']} {cc_map['n1_desc']}"
        nova["n2_cod_centro_custo"] = cc_map["n2_cod"]
        nova["n2_centro_custo"] = cc_map["n2_desc"]
        nova["n2_CC"] = f"{cc_map['n2_cod']} {cc_map['n2_desc']}"
        nova["n3_cod_centro_custo"] = cc_map["n3_cod"]
        nova["n3_centro_custo"] = cc_map["n3_desc"]
        nova["n3_CC"] = f"{cc_map['n3_cod']} {cc_map['n3_desc']}"
        nova["n4_cod_centro_custo"] = cc_map["n4_cod"]
        nova["n4_centro_custo"] = cc_map["n4_desc"]
        nova["n4_CC"] = f"{cc_map['n4_cod']} {cc_map['n4_desc']}"
        nova["filial"] = cc_map["filial_saida"]
    else:
        nova["filial"] = _str(row["nm_pessoa_filial"])

    if pc_map is not None:
        nova["cod_conta"] = pc_map["cod"]
        nova["conta"] = pc_map["desc"]
        nova["cod_conta-descr"] = f"{pc_map['cod']} {pc_map['desc']}"

    nova["titulo"] = _str(row["nr_documento"])
    nova["valor_nf"] = _fmt_brl(valor)
    nova["valor_pago"] = _fmt_brl(valor)
    nova["valor_conta"] = _fmt_brl(valor)
    nova["observacao"] = _str(row["nm_centro_custo"])
    nova["data_nf"] = _fmt_data(row["dt_lancamento_"])
    nova["data_pagamento"] = ""
    nova["credor_forn_cli_func"] = _str(row["nm_pessoa_favorecido"])
    nova["Dados auxiliares"] = _str(row["nm_historico"])
    nova["Origem"] = "Saidas (Aplicacoes)"
    nova["Sistema"] = "ATUA"

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

print(f"\nLinhas base ATUA: {len(df_atua)}")
print(f"Linhas geradas: {len(fechamento_df)}")
if len(fechamento_df) != len(df_atua):
    raise ValueError(
        f"Divergencia de linhas: base={len(df_atua)} vs fechamento={len(fechamento_df)}."
    )
print(f"Linhas com CC nao mapeado: {len(alertas_linhas_cc)}")
print(f"Linhas com PC nao mapeado: {len(alertas_linhas_pc)}")
fechamento_df.head(10)

Colunas do modelo FECHAMENTO: ['id', 'Segmento', 'n1_cod_centro_custo', 'n1_centro_custo', 'n1_CC', 'n2_cod_centro_custo', 'n2_centro_custo', 'n2_CC', 'n3_cod_centro_custo', 'n3_centro_custo', 'n3_CC', 'n4_cod_centro_custo', 'n4_centro_custo', 'n4_CC', 'cod_conta', 'conta', 'cod_conta-descr', 'filial', 'titulo', 'valor_nf', 'valor_pago', 'valor_conta', 'observacao', 'data_nf', 'data_pagamento', 'cod_credor_forn_cli_func', 'credor_forn_cli_func', 'Origem', 'Sistema', 'Dados auxiliares', ' Valor Oficial ', 'DE-PARA1', 'DE-PARA2', 'CUSTEIO VARIÁVEL']

Linhas base ATUA: 140
Linhas geradas: 140
Linhas com CC nao mapeado: 0
Linhas com PC nao mapeado: 4


,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,n3_CC,n4_cod_centro_custo,n4_centro_custo,n4_CC,cod_conta,...,valor_nf,valor_pago,valor_conta,observacao,data_nf,data_pagamento,cod_credor_forn_cli_func,credor_forn_cli_func,Origem,Sistema,Dados auxiliares,Valor Oficial,DE-PARA1,DE-PARA2,CUSTEIO VARIÁVEL
0,1,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"97,97","97,97","97,97",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
1,2,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"109,56","109,56","109,56",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
2,3,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"46,03","46,03","46,03",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
3,4,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"124,24","124,24","124,24",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
4,5,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"159,92","159,92","159,92",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
5,6,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"149,96","149,96","149,96",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
6,7,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"138,14","138,14","138,14",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
7,8,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"104,30","104,30","104,30",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
8,9,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"71,80","71,80","71,80",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>
9,10,TRANSMOVE GSL,1.4,DESPESA,1.4 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"3,42","3,42","3,42",FROTA TERCEIRO PRUDENTE,05/01/2026,,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,Saidas (Aplicacoes),ATUA,IMPOSTOS E TAXAS DIVERSAS,<NA>,<NA>,<NA>,<NA>


## Salvamento e relatorio

Gera o Excel final em `02-Referencias/ATUA/ATUA_despesas_fechamento_{MM}-{AAAA}.xlsx` e imprime o relatorio de itens nao mapeados para revisao manual.

In [34]:
import datetime
from openpyxl.styles import Font

SHEET_NAME = "ATUA_despesas_fechamento"

def _salvar_excel(destino, df):
    with pd.ExcelWriter(destino, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=SHEET_NAME, index=False)
        ws = writer.book[SHEET_NAME]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font

arquivo_saida_exec = ARQUIVO_SAIDA
try:
    _salvar_excel(arquivo_saida_exec, fechamento_df)
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
except PermissionError:
    ts = datetime.datetime.now().strftime("%H%M%S")
    arquivo_saida_exec = ARQUIVO_SAIDA.with_stem(f"{ARQUIVO_SAIDA.stem}_{ts}")
    _salvar_excel(arquivo_saida_exec, fechamento_df)
    print(
        f"[AVISO] Arquivo principal em uso ({ARQUIVO_SAIDA.name}). "
        f"Salvo como: {arquivo_saida_exec.resolve()}"
    )

print(f"Linhas gravadas: {len(fechamento_df)}")

if ccs_nao_mapeados_cc:
    print("\n[REVISAO MANUAL] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")

if historicos_nao_mapeados:
    print("\n[REVISAO MANUAL] Historicos (Plano de Contas) nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")

if not ccs_nao_mapeados_cc and not historicos_nao_mapeados:
    print("\nNenhum item pendente. Todos os centros de custo e planos de conta foram mapeados.")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\ATUA_despesas_fechamento_Jan-Fev.xlsx
Linhas gravadas: 140

[REVISAO MANUAL] Historicos (Plano de Contas) nao mapeados:
  cd_historico=31 (MULTA - OUTRAS)
